# 02 - State minimaps

Draws each card's state outline with a marker at the city's location, using Census cartographic boundary shapefiles. Output goes to `outputs/minimaps/` (not committed).

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import pandas as pd
from shapely.geometry import Point
import matplotlib.colors as mcolors

# Define the paths to your shapefiles
all_states_shapefile_path = '../data/raw/shapefiles/cb_2018_us_state_500k.shp'
alaska_shapefile_path = '../data/raw/shapefiles/Alaska.shp'
hawaii_shapefile_path = '../data/raw/shapefiles/Hawaii.shp'

# Specify the path to your US cities CSV file
csv_path = '../data/raw/minimap_colors_city_list_set_1.csv'

# Specify the directory where you want to save the output images
output_directory = '../outputs/minimaps'

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Load the main states shapefile
try:
    all_states = gpd.read_file(all_states_shapefile_path)
    all_states = all_states.to_crs(epsg=3857)
    print("Main states shapefile imported and reprojected successfully!")
except FileNotFoundError:
    print(f"Error: Shapefile not found at {all_states_shapefile_path}")
    exit()

# Load Alaska shapefile
try:
    alaska = gpd.read_file(alaska_shapefile_path)
    alaska = alaska.to_crs(epsg=3857)
    print("Alaska shapefile imported and reprojected successfully!")
except FileNotFoundError:
    print(f"Warning: Alaska shapefile not found at {alaska_shapefile_path}. Alaska plots might not be generated.")
    alaska = None

# Load Hawaii shapefile
try:
    hawaii = gpd.read_file(hawaii_shapefile_path)
    hawaii = hawaii.to_crs(epsg=3857)
    print("Hawaii shapefile imported and reprojected successfully!")
except FileNotFoundError:
    print(f"Warning: Hawaii shapefile not found at {hawaii_shapefile_path}. Hawaii plots might not be generated.")
    hawaii = None

# Load the cities .csv
try:
    cities_df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: CSV file not found at {csv_path}")
    exit()

# Iterate through each city in the cities DataFrame
for index, city_row in cities_df.iterrows():
    city_name = city_row['City']
    state_name = city_row['State']
    longitude = city_row['Longitude']
    latitude = city_row['Latitude']
    state_id = city_row['State ID']
    primary_color_hex = city_row['Primary']
    secondary_color_hex = city_row['Secondary']

    # Determine which shapefile to use based on the state
    if state_name == 'Alaska' and alaska is not None:
        state_gdf = alaska
    elif state_name == 'Hawaii' and hawaii is not None:
        state_gdf = hawaii
    else:
        state_gdf = all_states

    # Find the corresponding state geometry
    try:
        geometry = state_gdf[state_gdf['NAME'] == state_name].geometry.iloc[0]
    except IndexError:
        print(f"Warning: State '{state_name}' not found in the shapefiles for city '{city_name}'. Skipping.")
        continue

    # Calculate the bounding box of the geometry
    bounds = geometry.bounds

    # Determine the center of the bounding box
    center_x = (bounds[0] + bounds[2]) / 2
    center_y = (bounds[1] + bounds[3]) / 2

    # Find the maximum extent (width or height) of the bounding box
    max_extent = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    # Define the square plot extent, adding a small buffer
    buffer = 0.1 * max_extent  # Add a 10% buffer
    xlim = (center_x - max_extent / 2 - buffer, center_x + max_extent / 2 + buffer)
    ylim = (center_y - max_extent / 2 - buffer, center_y + max_extent / 2 + buffer)

    # Create a figure and axes for the plot
    fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=1200, facecolor='none')  # figsize in inches, dpi for pixel density

    # Determine state color
    if pd.notna(primary_color_hex):
        hex_color = primary_color_hex.strip()  # Remove any leading/trailing whitespace
        if not hex_color.startswith('#'):
            hex_color = '#' + hex_color
        try:
            state_color = mcolors.to_rgb(hex_color)
        except ValueError:
            print(f"Warning: Invalid primary color HEX code '{primary_color_hex}' for {city_name}, {state_name}. Defaulting to gainsboro.")
            state_color = 'gainsboro'
    else:
        state_color = 'gainsboro'

    # Plot the state geometry with the determined color
    gpd.GeoSeries([geometry]).plot(ax=ax, color=state_color, edgecolor=None)

    # Determine marker style and color
    marker_style = 'o'
    marker_color = 'white'
    #if pd.notna(secondary_color_hex):
    #    hex_color = secondary_color_hex.strip()  # Remove any leading/trailing whitespace
    #    if not hex_color.startswith('#'):
    #        hex_color = '#' + hex_color
    #    try:
    #        marker_color = mcolors.to_rgb(hex_color)
    #    except ValueError:
    #        print(f"Warning: Invalid secondary color HEX code '{secondary_color_hex}' for {city_name}, {state_name}. Defaulting to red.")
    #        marker_color = 'red'
    #else:
    #    marker_color = 'red'
    marker_size =  25
    marker_edgecolor = 'black'
    marker_linewidth = 0.5

    # Plot the city location, ensuring it's in the correct CRS
    city_gdf = gpd.GeoDataFrame(
        {'city': [city_name]},
        geometry=gpd.points_from_xy([longitude], [latitude]),
        crs="EPSG:4326"  # Assuming your CSV coordinates are in WGS84
    )
    city_gdf_reprojected = city_gdf.to_crs(state_gdf.crs)
    ax.scatter(
        city_gdf_reprojected.geometry.x, 
        city_gdf_reprojected.geometry.y, 
        color=marker_color, 
        marker=marker_style, 
        s=marker_size, 
        edgecolor=marker_edgecolor,
        linewidths=marker_linewidth,
        zorder=2 
    )

    # Set the x and y limits to ensure a square aspect ratio and include the state
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Remove axes ticks and labels for a cleaner look
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')

    # Set the aspect ratio to equal to ensure the state isn't stretched
    ax.set_aspect('equal', adjustable='box')

    # Construct the output filename using the state ID and city name
    filename = os.path.join(output_directory, f"{state_id}_{city_name.replace(' ', '_')}.png")

    # Save the figure as a PNG image with the desired resolution
    plt.savefig(filename, bbox_inches='tight', pad_inches=0, transparent=True)
    plt.close(fig)  # Close the figure to free up memory

    # print(f"Exported image for {city_name}, {state_name} to {filename}")

print("Done exporting images.")